# NOM / LSIP PHR Retest


- Uses the **exact linear objective** for LP/LSIP instead of a neural surrogate outside its training range.
- Uses a trainable displacement \(x=x_0+\Delta x\) rather than an over-parameterized full start layer.
- Uses **L-BFGS** to solve each PHR inner subproblem more accurately.
- Keeps adaptive penalty updates and constraint multipliers.
- Uses SciPy/HiGHS **only as an independent benchmark**, never to guide training.


## Imports and precision setup

In [1]:
import math
import time
import numpy as np
import torch
from scipy.optimize import linprog

torch.set_default_dtype(torch.float64)

SEED = 42

## Expression evaluator

In [2]:
# ============================================================
# Expression evaluation
# ============================================================

def eval_expr(expr, t):
    """Evaluate simple t-dependent expressions used in the LSIP examples."""
    allowed = {
        "t": t,
        "exp": torch.exp,
        "sin": torch.sin,
        "cos": torch.cos,
        "tan": torch.tan,
        "sqrt": torch.sqrt,
        "pi": math.pi,
    }
    return eval(expr, {"__builtins__": {}}, allowed)

## Problem builders

In [3]:
# ============================================================
# Problem builders
# ============================================================

def chebyshev_problem(power, m=1001):
    """
    Approximate t^power by a polynomial of degree power-1.

    Variable order:
        h, x1, ..., x_power
    """
    n = power + 1

    # Convert
    #   h + p(t) >= t^power
    #   h - p(t) >= -t^power
    # into A(t)x <= b(t).
    fam1 = ["-1", "-1"] + [
        "-t" if k == 1 else f"-t**{k}"
        for k in range(1, power)
    ]
    fam2 = ["-1", "1"] + [
        "t" if k == 1 else f"t**{k}"
        for k in range(1, power)
    ]

    return {
        "name": f"Chebyshev t^{power}",
        "n_vars": n,
        "sense": "min",
        "objective_coeffs": np.array([1.0] + [0.0] * power),
        "t_min": 0.0,
        "t_max": 1.0,
        "m": m,
        "start_interval": (0.0, 1.0),
        "families": [
            {
                "coeff_exprs": fam1,
                "rhs_expr": f"-t**{power}",
            },
            {
                "coeff_exprs": fam2,
                "rhs_expr": f"t**{power}",
            },
        ],
    }


def tan_problem(m=1001):
    """
    min x1 + 1/2 x2 + 1/3 x3
    s.t. x1 + t x2 + t^2 x3 >= tan(t)
    """
    return {
        "name": "One-sided tan(t)",
        "n_vars": 3,
        "sense": "min",
        "objective_coeffs": np.array([1.0, 0.5, 1.0 / 3.0]),
        "t_min": 0.0,
        "t_max": 1.0,
        "m": m,
        "start_interval": (0.0, 2.0),
        "families": [
            {
                "coeff_exprs": ["-1", "-t", "-t**2"],
                "rhs_expr": "-tan(t)",
            }
        ],
    }

## Constraint construction

In [4]:
# ============================================================
# Constraint construction
# ============================================================

def build_matrix(problem, m=None):
    if m is None:
        m = problem["m"]

    t = torch.linspace(
        problem["t_min"],
        problem["t_max"],
        m,
        dtype=torch.float64,
    )

    rows_A = []
    rows_b = []

    for fam in problem["families"]:
        cols = []
        for expr in fam["coeff_exprs"]:
            val = eval_expr(expr, t)
            if not torch.is_tensor(val):
                val = torch.full_like(t, float(val))
            elif val.ndim == 0:
                val = torch.full_like(t, float(val.item()))
            cols.append(val)

        A_f = torch.column_stack(cols)

        b_f = eval_expr(fam["rhs_expr"], t)
        if not torch.is_tensor(b_f):
            b_f = torch.full_like(t, float(b_f))
        elif b_f.ndim == 0:
            b_f = torch.full_like(t, float(b_f.item()))

        rows_A.append(A_f)
        rows_b.append(b_f)

    return torch.cat(rows_A, dim=0), torch.cat(rows_b, dim=0)

## Independent LP benchmark

In [5]:
# ============================================================
# Independent LP benchmark
# ============================================================

def solve_lp(problem, A, b):
    c = np.asarray(problem["objective_coeffs"], dtype=float)

    out = linprog(
        c,
        A_ub=A.detach().cpu().numpy(),
        b_ub=b.detach().cpu().numpy(),
        bounds=[(None, None)] * problem["n_vars"],
        method="highs",
    )

    if not out.success:
        raise RuntimeError(out.message)

    return float(out.fun), np.asarray(out.x, dtype=float)

## Dense-grid validation

In [6]:
# ============================================================
# Dense original-LSIP validation
# ============================================================

def dense_violation(problem, x, dense_m=100001):
    A, b = build_matrix(problem, m=dense_m)
    x_t = torch.as_tensor(x, dtype=torch.float64)
    g = A @ x_t - b

    idx = int(torch.argmax(g).item())
    max_v = max(0.0, float(g[idx].item()))

    # Families are stacked one after another.
    local_idx = idx % dense_m
    t_grid = np.linspace(problem["t_min"], problem["t_max"], dense_m)

    return max_v, float(t_grid[local_idx])

## Corrected PHR-NOM solver

In [7]:
# ============================================================
# Corrected PHR-NOM retest
# ============================================================

def phr_nom_retest(
    problem,
    n_starts=3,
    outer_iterations=25,
    inner_max_iter=100,
    rho0=1.0,
    rho_max=100000.0,
    rho_multiplier=5.0,
    feasibility_tol=1e-6,
):
    """
    NOM-compatible PHR diagnostic.

    A fixed starting point x0 is retained.
    Instead of an over-parameterized n x n linear map for one sample,
    use a trainable displacement delta:

        x = x0 + delta

    This keeps the starting-point optimization idea while giving one
    trainable degree of freedom per decision variable.

    The objective is exactly c^T x because the problems are linear.
    """

    A, b = build_matrix(problem)

    c = torch.tensor(
        problem["objective_coeffs"],
        dtype=torch.float64,
    )

    if problem["sense"] == "max":
        c = -c

    lp_obj, lp_x = solve_lp(problem, A, b)

    low, high = problem["start_interval"]

    rng = torch.Generator().manual_seed(SEED)
    starts = (
        torch.rand(
            (n_starts, problem["n_vars"]),
            generator=rng,
            dtype=torch.float64,
        )
        * (high - low)
        + low
    )

    best_global = None
    all_rows = []

    start_time = time.perf_counter()

    for start_id in range(n_starts):
        x0 = starts[start_id].clone()

        # Trainable NOM displacement.
        delta = torch.nn.Parameter(
            torch.zeros(problem["n_vars"], dtype=torch.float64)
        )

        lam = torch.zeros(len(b), dtype=torch.float64)
        rho = float(rho0)
        previous_violation = float("inf")

        best_start = None

        for outer in range(outer_iterations):

            # PHR requires reasonably accurate minimization of the
            # current augmented-Lagrangian subproblem.
            optimizer = torch.optim.LBFGS(
                [delta],
                lr=1.0,
                max_iter=inner_max_iter,
                tolerance_grad=1e-11,
                tolerance_change=1e-13,
                line_search_fn="strong_wolfe",
            )

            def closure():
                optimizer.zero_grad()

                x = x0 + delta
                g = A @ x - b

                shifted = torch.relu(lam + rho * g)

                phr_term = (
                    shifted.square() - lam.square()
                ).sum() / (2.0 * rho)

                loss = c @ x + phr_term
                loss.backward()

                return loss

            optimizer.step(closure)

            with torch.no_grad():
                x = x0 + delta
                g = A @ x - b

                violation = max(
                    0.0,
                    float(g.max().item()),
                )

                transformed_obj = float((c @ x).item())

                if problem["sense"] == "max":
                    original_obj = -transformed_obj
                else:
                    original_obj = transformed_obj

                candidate = {
                    "start": start_id,
                    "outer": outer,
                    "rho": rho,
                    "objective": original_obj,
                    "violation": violation,
                    "x": x.detach().cpu().numpy().copy(),
                }

                if violation <= feasibility_tol:
                    better = (
                        best_start is None
                        or original_obj < best_start["objective"]
                    )
                    if problem["sense"] == "max":
                        better = (
                            best_start is None
                            or original_obj > best_start["objective"]
                        )

                    if better:
                        best_start = candidate.copy()

                    better_global = (
                        best_global is None
                        or original_obj < best_global["objective"]
                    )
                    if problem["sense"] == "max":
                        better_global = (
                            best_global is None
                            or original_obj > best_global["objective"]
                        )

                    if better_global:
                        best_global = candidate.copy()

                # PHR multiplier update.
                lam = torch.relu(lam + rho * g)

                # Increase rho only if feasibility improvement stalls.
                if (
                    violation > feasibility_tol
                    and np.isfinite(previous_violation)
                    and violation > 0.5 * previous_violation
                ):
                    rho = min(
                        rho * rho_multiplier,
                        rho_max,
                    )

                previous_violation = violation

        if best_start is None:
            with torch.no_grad():
                x = x0 + delta
                g = A @ x - b
                obj_t = float(c @ x)
                best_start = {
                    "start": start_id,
                    "outer": outer_iterations - 1,
                    "rho": rho,
                    "objective": -obj_t if problem["sense"] == "max" else obj_t,
                    "violation": max(0.0, float(g.max().item())),
                    "x": x.detach().cpu().numpy().copy(),
                }

        all_rows.append(best_start)

    runtime = time.perf_counter() - start_time

    if best_global is None:
        # Fallback only if no point satisfies the chosen tolerance.
        best_global = min(
            all_rows,
            key=lambda r: r["violation"],
        )

    dense_v, dense_t = dense_violation(
        problem,
        best_global["x"],
    )

    gap = abs(best_global["objective"] - lp_obj)
    rel_error = gap / abs(lp_obj) * 100.0

    x_distance = float(
        np.linalg.norm(best_global["x"] - lp_x)
    )

    return {
        "problem": problem["name"],
        "lp_objective": lp_obj,
        "lp_x": lp_x,
        "phr_objective": best_global["objective"],
        "phr_x": best_global["x"],
        "absolute_gap": gap,
        "relative_error_percent": rel_error,
        "sampled_violation": best_global["violation"],
        "dense_violation": dense_v,
        "dense_worst_t": dense_t,
        "x_distance": x_distance,
        "selected_start": best_global["start"],
        "selected_outer": best_global["outer"],
        "selected_rho": best_global["rho"],
        "runtime_sec": runtime,
        "all_starts": all_rows,
    }

## Result display helper

In [8]:
# ============================================================
# Display
# ============================================================

def print_result(r):
    print("\n" + "=" * 76)
    print(r["problem"])
    print("=" * 76)

    print("LP objective       :", r["lp_objective"])
    print("Corrected PHR obj  :", r["phr_objective"])
    print("Absolute gap       :", r["absolute_gap"])
    print("Relative error (%) :", r["relative_error_percent"])

    print("Sample violation   :", r["sampled_violation"])
    print("Dense violation    :", r["dense_violation"])
    print("Dense worst t      :", r["dense_worst_t"])

    print("x-distance vs LP   :", r["x_distance"])
    print("Selected start     :", r["selected_start"])
    print("Selected outer     :", r["selected_outer"])
    print("Selected rho       :", r["selected_rho"])
    print("Runtime (s)        :", r["runtime_sec"])

    print("\nLP x:")
    print(r["lp_x"])

    print("\nCorrected PHR x:")
    print(r["phr_x"])

## Run all benchmark problems

In [9]:
tests = [
    tan_problem(),
    chebyshev_problem(3),
    chebyshev_problem(4),
    chebyshev_problem(5),
    chebyshev_problem(6),
]

results = []

for problem in tests:
    result = phr_nom_retest(problem)
    results.append(result)
    print_result(result)



One-sided tan(t)
LP objective       : 0.6490419836878412
Corrected PHR obj  : 0.649041538846648
Absolute gap       : 4.448411932145291e-07
Relative error (%) : 6.853812301739743e-05
Sample violation   : 6.737158186331804e-07
Dense violation    : 7.392073101408769e-07
Dense worst t      : 0.33332
x-distance vs LP   : 0.0006166647671421629
Selected start     : 1
Selected outer     : 14
Selected rho       : 100000.0
Runtime (s)        : 56.754964299994754

LP x:
[0.08920578 0.42261335 1.0455886 ]

Corrected PHR x:
[0.08908412 0.42309731 1.0452263 ]

Chebyshev t^3
LP objective       : 0.03125000000000008
Corrected PHR obj  : 0.031249593947183248
Absolute gap       : 4.0605281683564476e-07
Relative error (%) : 0.0012993690138740598
Sample violation   : 6.096083945278385e-07
Dense violation    : 6.096083945278385e-07
Dense worst t      : 0.25
x-distance vs LP   : 9.809352085869716e-07
Selected start     : 2
Selected outer     : 13
Selected rho       : 78125.0
Runtime (s)        : 58.7920356